In [ ]:
"""
lda_nmf_comparison.py  05JUN2026
================================================================================
LDA vs NMF comparison at k = 15, run under three RAKE configurations.

This cell carries the small set of pipeline helpers it needs (stopwords, the a-priori codebook, text cleaning/tokenization, the data
loader, RAKE, codebook alignment, top-word extraction, and figure styling), so it can be run on its own without first executing the main pipeline cell.

The three configurations answer "where should RAKE sit relative to the topic model?", holding k fixed at 15 and comparing LDA against NMF in each:

  (1) RAKE as PREPROCESSING
        Corpus-level RAKE phrases are merged into atomic tokens (e.g. "magaling mag explain" -> "magaling_mag_explain") and injected into the
        modeled vocabulary BEFORE fitting, so both LDA and NMF see the phrase as one feature. Tests whether feeding RAKE phrases in as features changes
        topic quality / cross-method agreement.

  (2) NO RAKE  (baseline multi-model evaluation)
        Standard TF-IDF/NMF and BoW/LDA on the existing tokens. The clean head-to-head LDA-vs-NMF reference the other two configs are judged against.

  (3) RAKE as POST-PROCESSING  (topic labeling)
        LDA and NMF are fit WITHOUT RAKE (the baseline models from config 2), then RAKE is run on each discovered topic's assigned comments to produce a
        human-readable multi-word LABEL per topic. RAKE never touches discovery here; it only names topics after the fact.

Outputs (written under {OUTPUT_ROOT}/k15_lda_nmf_comparison/):
    comparison_metrics.csv        — coherence (c_v), topic diversity, codebook
                                    alignment counts, per (config x model)
    topic_keywords_by_model.csv   — top words per topic per model per config
    cross_method_agreement.csv    — greedy LDA<->NMF topic matches (Jaccard) per config
    topic_labels_rake.csv         — config-3 RAKE labels, LDA and NMF side by side
    topic_codebook_alignment.csv  — per-topic aligned/ambiguous/inductive status
    figures/cmp_coherence.png     — coherence by config x model
    figures/cmp_alignment.png     — codebook alignment mix by config x model
    figures/cmp_agreement.png     — cross-method (LDA vs NMF) agreement by config

Requirements:  pip install scikit-learn gensim nltk matplotlib pandas numpy
"""

import os
import re
import warnings
import logging
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING)

# ── Configuration ─────────────────────────────────────────────────────────────
# Same source/output conventions as the main pipeline. Comparison artefacts are
# written under {OUTPUT_ROOT}/k15_lda_nmf_comparison/.
DATA_DIR    = "/content/drive/MyDrive/SEAAIR2026/code and dataset"
TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = os.path.join(DATA_DIR, "thematic_analysis", TIMESTAMP, "outputs")

MIN_WORDS   = 8          # exclude very short comments (matches pipeline)
MAX_VOCAB   = 5000       # TF-IDF/CountVec vocabulary ceiling
LDA_PASSES  = 15         # more passes = better coherence, slower
LDA_ITER    = 400
RANDOM_SEED = 42


# ════════════════════════════════════════════════════════════════════════════════
# PIPELINE HELPERS (carried in so this cell is standalone)
# ════════════════════════════════════════════════════════════════════════════════

# ── NLTK data ─────────────────────────────────────────────────────────────────
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

# ── Color palette (per HEI + tie type) ───────────────────────────────────────
SUB_C = {
    "Benilde": "#88E788",
    "dlsu":    "#198754",
    "peyups":  "#dc3545",
    "AdMU":    "#0d6efd",
    "unknown": "#adb5bd",
}
TIE_C = {"weak": "#fd7e14", "strong": "#4dabf7"}
BG, SF, TX, MU, GR = "#FFFFFF", "#F8F9FA", "#212529", "#6C757D", "#DEE2E6"
SAVE = dict(dpi=180, bbox_inches="tight", facecolor=BG, edgecolor="none")

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": SF, "axes.edgecolor": GR,
    "text.color": TX, "axes.labelcolor": TX,
    "xtick.color": MU, "ytick.color": MU,
    "grid.color": GR, "grid.linewidth": 0.6,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False, "axes.spines.right": False,
})

# ── Bot list ──────────────────────────────────────────────────────────────────
# Mirrors 01_load_and_clean.py (Module 1) exactly. Only used on the raw-file
# fallback path; when the canonical cleaned/comments_clean.csv is read, cleaning
# has already been applied upstream and this list is not consulted.
BOTS = {
    "AutoModerator", "BotDefense", "RepostSleuthBot", "RemindMeBot",
    "MAGIC_EYE_BOT", "anti-gif-bot", "Bot_Metric", "reddit-bot",
    "Deleted", "deleted", "[deleted]",
}

# ── Filipino + English stopwords combined ─────────────────────────────────────
FIL_STOPS = {
    "ang", "ng", "na", "sa", "at", "ay", "ni", "mga", "kung", "ito",
    "ko", "mo", "ka", "po", "din", "rin", "nga", "ba", "si", "naman",
    "yung", "yun", "ung", "yon", "siya", "niya", "nila", "kami", "tayo",
    "kayo", "sila", "ako", "ikaw", "kasi", "pero", "lang", "talaga",
    "daw", "raw", "pala", "natin", "namin", "ninyo", "wala", "may",
    "mayroon", "oo", "opo", "hindi", "huwag", "para", "saan", "paano",
    "bakit", "kailan", "sino", "ano", "eh", "kaya", "pag", "kapag",
    "kahit", "habang", "bago", "pagkatapos", "ha", "haha", "hehe",
    "lol", "lmao", "hmm", "hm", "ah", "oh", "yep", "yeah", "ok",
    "okay", "im", "ive", "id", "dont", "doesnt", "didnt", "isnt",
    "wasnt", "arent", "weren", "ive", "weve", "theyre", "theyll",
    "its", "thats", "isnt", "arent", "wasnt", "shouldnt", "couldnt",
    "wouldnt", "nd", "ta", "di", "mas", "rin",
}
EN_STOPS = set(stopwords.words("english"))
ALL_STOPS = FIL_STOPS | EN_STOPS | {
    # Reddit-specific noise
    "deleted", "removed", "edit", "reddit", "post", "thread", "comment",
    "subreddit", "upvote", "downvote", "mod", "moderator", "bot",
    "http", "https", "www", "com", "link", "url", "image", "gif",
    "crosspost", "repost", "oc", "tldr", "tl", "dr", "edt",
    # Generic filler
    "really", "just", "also", "even", "would", "could", "should",
    "will", "can", "get", "got", "going", "said", "know", "think",
    "much", "still", "well", "back", "around", "though", "already",
    "actually", "probably", "maybe", "something", "someone", "people",
    "thing", "things", "way", "ways", "lot", "lots", "bit", "bit",
    "thank", "thanks", "please", "ask", "sure", "try", "need", "want",
    "hope", "say", "tell", "make", "made", "look", "looks", "feel",
}

# ── Preprocessing ─────────────────────────────────────────────────────────────
_lemmatizer = WordNetLemmatizer()


def clean_text(text: str) -> str:
    """Normalize Reddit comment text for tokenization."""
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Remove Reddit mention patterns (u/user, r/sub)
    text = re.sub(r"\bu/\w+", " ", text)
    text = re.sub(r"\br/\w+", " ", text)
    # Remove special characters but keep apostrophes for contractions
    text = re.sub(r"[^a-z\s']", " ", text)
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    """Tokenize, lemmatize, and remove stopwords."""
    tokens = clean_text(text).split()
    tokens = [
        _lemmatizer.lemmatize(t)
        for t in tokens
        if t.isalpha()
        and len(t) > 2
        and t not in ALL_STOPS
    ]
    return tokens

# ── Codebook (a priori template — used for POST-HOC alignment only) ───────────
# These seeds do NOT discover topics. NMF/LDA run fully unsupervised; the codebook
# is applied AFTER discovery to measure how each emergent topic aligns with the
# four-domain qualitative codebook. This implements the hybrid inductive-deductive
# design (Fereday & Muir-Cochrane, 2006; a priori template, Crabtree & Miller,
# 1999): alignment is REPORTED, never forced. Topics with no codebook match are
# surfaced as inductive candidates for the human coding pass, not relabelled.
#
# Rhetorical / epistemic subthemes (C1 evidence type, C2 rhetorical strategy,
# C3 specificity, D1 credibility norms) describe HOW evaluation is expressed, not
# which content words appear, so they are deliberately left unseeded (lexical=False).
# A bag-of-words model cannot recover them; their absence from computational
# alignment is by design and must NOT be read as a finding — they are carried by
# the human coding (see RQ2 protocol §4).

DOMAIN_LABELS = {
    "A": "Faculty Performance Evaluation",
    "B": "Institutional and Course Evaluation",
    "C": "Evaluative Rhetoric and Epistemic Framing",
    "D": "Community Norms and Relational Dynamics",
    "?": "Inductive candidate — no codebook match",
}

# code -> (label, domain, lexical?, seed terms)
CODEBOOK = {
    "A1": ("Teaching effectiveness", "A", True, [
        "teach", "teaching", "explain", "explains", "explained", "lecture",
        "lessons", "clear", "understand", "magaling", "galing", "knowledgeable",
        "engaging", "boring", "prepared", "learn", "learned", "discussion",
        "approachable"]),
    "A2": ("Grading and assessment", "A", True, [
        "grade", "grades", "grading", "exam", "exams", "quiz", "recit",
        "recitation", "rubric", "score", "fail", "failed", "passing", "output",
        "requirements", "curve", "bell", "lenient", "deadline", "points"]),
    "A3": ("Professional conduct", "A", True, [
        "late", "absent", "cancel", "respond", "reply", "email", "bias",
        "biased", "favoritism", "pabor", "terror", "strict", "rude",
        "respectful", "unfair", "fair", "unprofessional", "ghosting",
        "accommodating", "attitude"]),
    "A4": ("Recommendation/avoidance", "A", True, [
        "avoid", "recommend", "beware", "warning", "warn", "suggest", "advice",
        "choose", "pick", "iwasan", "kunin", "drop", "enroll"]),
    "B1": ("Course evaluation", "B", True, [
        "course", "subject", "subjects", "syllabus", "units", "load", "elective",
        "curriculum", "workload", "materials", "modules", "trimester",
        "semester", "prerequisite"]),
    "B2": ("Institutional policies", "B", True, [
        "tuition", "fees", "fee", "enrollment", "registrar", "admin",
        "administration", "policy", "policies", "registration", "system",
        "office", "scholarship", "financial"]),
    "B3": ("Comparative institutional evaluation", "B", True, [
        "dlsu", "ateneo", "admu", "benilde", "peyups", "lasalle", "salle",
        "ust", "mapua", "compared", "comparison", "versus", "kesa", "better"]),
    "C1": ("Evidence type", "C", False, []),
    "C2": ("Rhetorical strategy", "C", False, []),
    "C3": ("Evaluation specificity", "C", False, []),
    "C4": ("Platform reflexivity", "C", True, [
        "reddit", "subreddit", "sub", "thread", "threads", "upvote", "downvote",
        "karma", "repost", "mods", "mod"]),
    "D1": ("Evaluation credibility norms", "D", False, []),
    "D2": ("Solidarity and support", "D", True, [
        "support", "agree", "same", "true", "valid", "sorry", "kaya",
        "kakayanin", "laban", "congrats", "ingat", "fellow", "classmate",
        "batchmate", "sana", "goodluck"]),
    "D3": ("Gatekeeping and exclusion", "D", True, [
        "gatekeep", "elitist", "deserve", "entitled", "exclusive", "poser"]),
    "D4": ("Filipino cultural evaluation frames", "D", True, [
        "terror", "pabor", "bahala", "kulit", "diskarte", "sir", "maam", "ate",
        "kuya", "suki", "hiya", "utang", "padrino", "lusot", "palakasan",
        "iskolar", "mentor"]),
}

# Derived lookups
SEEDSETS = {c: set(seeds) for c, (lab, dom, lex, seeds) in CODEBOOK.items() if lex}
LEXICAL_SUBTHEMES    = [c for c, (lab, dom, lex, seeds) in CODEBOOK.items() if lex]
HUMAN_ONLY_SUBTHEMES = [c for c, (lab, dom, lex, seeds) in CODEBOOK.items() if not lex]

# Alignment thresholds (tunable)
MIN_ALIGN_OVERLAP = 2   # >=2 matched seed terms needed for a confident alignment
AMBIG_MARGIN      = 1    # best must beat a cross-domain runner-up by this margin

def load_data(sample_n: int | None = None) -> pd.DataFrame:
    """
    Load the comment corpus, preferring the canonical cleaned file produced by
    Module 1 (cleaned/comments_clean.csv) so that cleaning rules and the weak/
    strong tie_type are IDENTICAL to the sentiment and network strands.

    If the cleaned file is present, its existing tie_type column is reused as-is
    (single source of truth). Only when an uncleaned raw file is used as a
    fallback are bot/deleted filtering and the tie_type median split applied
    here, using the same bot list and rule as Module 1.

    The thematic-only steps (≥MIN_WORDS filter, and the tokenization in
    preprocess()) are applied on top regardless of source.
    """
    print("Loading data...")
    cleaned_path = os.path.join(DATA_DIR, "cleaned", "comments_clean.csv")
    raw_path     = os.path.join(DATA_DIR, "dataset_comments.csv")

    if os.path.exists(cleaned_path):
        comments_path, from_cleaned = cleaned_path, True
    elif os.path.exists(raw_path):
        comments_path, from_cleaned = raw_path, False
    else:
        raise FileNotFoundError(
            f"No input found. Expected {cleaned_path} (preferred) "
            f"or {raw_path} (raw fallback)."
        )

    c = pd.read_csv(comments_path, parse_dates=["comment_created_utc"])
    print(f"  Source: {comments_path}")

    if not from_cleaned:
        # Raw fallback only — replicate Module 1 cleaning so the corpus matches
        # the other strands. (Skipped when the cleaned file is used.)
        c = (c[~c["comment_author"].isin(BOTS) &
               ~c["comment_body"].isin(["[deleted]", "[removed]"])]
             .dropna(subset=["comment_body"])
             .drop_duplicates("comment_id")
             .copy())
    else:
        c = c.copy()

    # Tie type: reuse the canonical column if present; derive only if missing.
    if "tie_type" in c.columns and c["tie_type"].notna().any():
        print("  Using existing canonical tie_type (from Module 1).")
    else:
        med = c["tie_strength_proxy"].median()
        c["tie_type"] = c["tie_strength_proxy"].apply(
            lambda x: "weak" if x > med else "strong"
        )
        print(f"  tie_type column absent — derived locally "
              f"(median proxy = {med}, Module 1 rule).")

    # Temporal features: reuse if the cleaned file already has them, else compute.
    if "year_month" not in c.columns:
        c["year_month"] = c["comment_created_utc"].dt.to_period("M").astype(str)
    if "year" not in c.columns:
        c["year"] = c["comment_created_utc"].dt.year
    if "month" not in c.columns:
        c["month"] = c["comment_created_utc"].dt.month

    # Thematic-only: comment length and the substantive-comment filter.
    c["word_count"] = (c["comment_body"].astype(str)
                       .str.split().str.len().fillna(0).astype(int))
    c = c[c["word_count"] >= MIN_WORDS].copy()

    if sample_n:
        c = c.sample(min(sample_n, len(c)), random_state=RANDOM_SEED)

    print(f"  Loaded {len(c):,} substantive comments (≥{MIN_WORDS} words)")
    return c.reset_index(drop=True)


# ════════════════════════════════════════════════════════════════════════════════
# PREPROCESSING
# ════════════════════════════════════════════════════════════════════════════════
def preprocess(df: pd.DataFrame) -> tuple[list[list[str]], list[str]]:
    """Tokenize all comments. Returns token lists and joined strings."""
    print("Preprocessing...")
    token_lists, joined = [], []
    for text in df["comment_body"]:
        toks = tokenize(str(text))
        token_lists.append(toks)
        joined.append(" ".join(toks))
    return token_lists, joined

# ── Top-word extraction / RAKE / codebook alignment ───────────────────────────

def get_top_words(H: np.ndarray, feat_names: list[str], n: int = 12) -> list[list[str]]:
    """Extract top-n words for each NMF topic."""
    return [
        [feat_names[j] for j in row.argsort()[-n:][::-1]]
        for row in H
    ]

def rake_keywords(texts: list[str], top_n: int = 20) -> list[tuple[str, float]]:
    """
    Rapid Automatic Keyword Extraction (RAKE).
    Extracts multi-word phrases without a pre-trained model.
    """
    stop_set = ALL_STOPS

    def score_phrase(phrase: str) -> float:
        words = phrase.split()
        if len(words) == 0:
            return 0.0
        freq = Counter(words)
        degree = sum(len(phrase.split()) for phrase in [phrase])
        return sum((degree / freq[w]) for w in words)

    phrase_freq: Counter = Counter()
    phrase_score: dict = {}

    for text in texts:
        text_clean = clean_text(str(text))
        # Split on stop words to get candidate phrases
        stop_pattern = r"\b(?:" + "|".join(re.escape(s) for s in stop_set) + r")\b"
        phrases = re.split(stop_pattern, text_clean)
        for ph in phrases:
            ph = ph.strip()
            words = [w for w in ph.split() if w.isalpha() and len(w) > 2]
            if 1 <= len(words) <= 4:
                key = " ".join(words)
                phrase_freq[key] += 1
                phrase_score[key] = score_phrase(key) * phrase_freq[key]

    return sorted(phrase_score.items(), key=lambda x: -x[1])[:top_n]

def align_to_codebook(top_words: list[str], n_consider: int = 12) -> dict:
    """
    Post-hoc alignment of an ALREADY-discovered topic to the a priori codebook.

    This does not influence topic discovery (NMF/LDA are unsupervised). It measures
    overlap between a topic's keywords and each lexically-seeded subtheme, then
    reports the alignment transparently instead of forcing a nearest bucket:

      status = "aligned"             clear, multi-term match to one subtheme
             = "ambiguous"           weak (single term) or cross-domain near-tie
             = "inductive_candidate" no seed overlap -> feeds human inductive coding
    """
    words = list(dict.fromkeys([w for w in top_words[:n_consider]]))  # ordered, unique
    rank = {w: i for i, w in enumerate(words)}

    hits = []
    for code in LEXICAL_SUBTHEMES:
        matched = [w for w in words if w in SEEDSETS[code]]
        if matched:
            overlap = len(matched)
            weighted = round(sum(1.0 / (rank[w] + 1) for w in matched), 3)
            hits.append((code, overlap, weighted, matched))

    base = {
        "subtheme_code": "?", "subtheme_label": DOMAIN_LABELS["?"],
        "domain_code": "?",   "domain_label": DOMAIN_LABELS["?"],
        "status": "inductive_candidate",
        "overlap": 0, "alignment_strength": 0.0, "matched_terms": "",
        "runner_up_code": "", "runner_up_overlap": 0, "runner_up_domain": "",
    }
    if not hits:
        return base

    hits.sort(key=lambda h: (h[1], h[2]), reverse=True)
    best = hits[0]
    runner = hits[1] if len(hits) > 1 else None

    b_code, b_overlap, b_weighted, b_matched = best
    b_label, b_domain, _, _ = CODEBOOK[b_code]

    r_code = r_overlap = r_domain = None
    clear = True
    if runner is not None:
        r_code, r_overlap, r_weighted, _ = runner
        _, r_domain, _, _ = CODEBOOK[r_code]
        if r_domain != b_domain and (b_overlap - r_overlap) < AMBIG_MARGIN:
            clear = False  # a different domain is essentially tied for best

    if b_overlap >= MIN_ALIGN_OVERLAP and clear:
        status = "aligned"
    elif b_overlap >= 1:
        status = "ambiguous"
    else:
        status = "inductive_candidate"

    inductive = status == "inductive_candidate"
    return {
        "subtheme_code":  "?" if inductive else b_code,
        "subtheme_label": DOMAIN_LABELS["?"] if inductive else b_label,
        "domain_code":    "?" if inductive else b_domain,
        "domain_label":   DOMAIN_LABELS["?"] if inductive else DOMAIN_LABELS[b_domain],
        "status":         status,
        "overlap":        b_overlap,
        "alignment_strength": round(b_overlap / max(1, len(words)), 2),
        "matched_terms":  ", ".join(b_matched),
        "runner_up_code":    r_code or "",
        "runner_up_overlap": r_overlap or 0,
        "runner_up_domain":  r_domain or "",
    }

# ════════════════════════════════════════════════════════════════════════════════
# COMPARISON HARNESS
# ════════════════════════════════════════════════════════════════════════════════
# Fixed parameters for this comparison
# Fixed parameters for this comparison
CMP_K          = 15     # topic count held constant across every run
TOP_N_WORDS    = 12     # top words per topic used for coherence/alignment/matching
RAKE_VOCAB_N   = 250    # number of corpus-level RAKE phrases injected in config 1
RAKE_LABEL_N   = 1      # RAKE phrases kept as the label per topic in config 3


# ════════════════════════════════════════════════════════════════════════════════
# SMALL METRIC HELPERS
# ════════════════════════════════════════════════════════════════════════════════
def _expand_phrase_tokens(words: list[str]) -> list[str]:
    """
    Split underscore-joined RAKE phrase tokens back into their component words so
    codebook seed-matching and cross-method Jaccard treat 'magaling_explain' as
    {'magaling', 'explain'}. Order preserved, duplicates dropped.
    """
    out: list[str] = []
    for w in words:
        for piece in str(w).split("_"):
            piece = piece.strip()
            if piece and piece not in out:
                out.append(piece)
    return out


def _lda_top_words(model: LdaModel, n: int = TOP_N_WORDS) -> list[list[str]]:
    """Top-n words for every LDA topic, in topic-id order."""
    return [[w for w, _ in model.show_topic(t, topn=n)]
            for t in range(model.num_topics)]


def _coherence_cv(topics_words: list[list[str]],
                  texts: list[list[str]],
                  dictionary: corpora.Dictionary) -> float:
    """
    c_v coherence for an arbitrary set of topic word-lists (works for NMF too,
    not just gensim LDA models). Words absent from `dictionary` are dropped, and
    topics left with < 2 scorable words are skipped so a stray out-of-vocab term
    cannot crash the scorer.
    """
    vocab = dictionary.token2id
    filtered = [[w for w in tw if w in vocab] for tw in topics_words]
    filtered = [tw for tw in filtered if len(tw) >= 2]
    if not filtered:
        return float("nan")
    try:
        return CoherenceModel(
            topics=filtered, texts=texts,
            dictionary=dictionary, coherence="c_v",
        ).get_coherence()
    except Exception:
        return float("nan")


def _topic_diversity(topics_words: list[list[str]], n: int = TOP_N_WORDS) -> float:
    """
    Topic diversity (Dieng et al., 2020): share of unique words across the top-n
    of all topics. 1.0 = every topic word distinct; low = topics repeat the same
    terms (a common NMF/LDA degeneracy worth comparing across configs).
    """
    flat = [w for tw in topics_words for w in tw[:n]]
    if not flat:
        return float("nan")
    return round(len(set(flat)) / len(flat), 3)


def _jaccard(a: list[str], b: list[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def _match_topics(lda_words: list[list[str]],
                  nmf_words: list[list[str]]) -> tuple[float, list[dict]]:
    """
    Greedily match each NMF topic to its most similar LDA topic by top-word
    Jaccard (phrases expanded first). Returns the mean best-match Jaccard — a
    single 0-1 "do the two methods recover the same structure?" score — plus the
    matched pairs for the per-config detail CSV.
    """
    lda_exp = [_expand_phrase_tokens(w) for w in lda_words]
    nmf_exp = [_expand_phrase_tokens(w) for w in nmf_words]

    used_lda: set[int] = set()
    pairs: list[dict] = []
    for ni, nw in enumerate(nmf_exp):
        best_j, best_l = -1.0, -1
        for li, lw in enumerate(lda_exp):
            if li in used_lda:
                continue
            j = _jaccard(nw, lw)
            if j > best_j:
                best_j, best_l = j, li
        if best_l >= 0:
            used_lda.add(best_l)
        pairs.append({
            "nmf_topic":   ni,
            "lda_topic":   best_l,
            "jaccard":     round(best_j, 3),
            "shared_terms": ", ".join(
                sorted(set(nmf_exp[ni]) & set(lda_exp[best_l]))
            ) if best_l >= 0 else "",
            "nmf_top": ", ".join(nmf_words[ni][:6]),
            "lda_top": ", ".join(lda_words[best_l][:6]) if best_l >= 0 else "",
        })
    mean_j = round(float(np.mean([p["jaccard"] for p in pairs])), 3) if pairs else 0.0
    return mean_j, pairs


def _alignment_counts(topics_words: list[list[str]]) -> tuple[dict, list[dict]]:
    """
    Run the post-hoc codebook alignment on each topic (phrases expanded so RAKE
    multi-word features can still match single-word seeds) and tally how many
    topics land as aligned / ambiguous / inductive.
    """
    rows, counts = [], {"aligned": 0, "ambiguous": 0, "inductive_candidate": 0}
    for i, words in enumerate(topics_words):
        a = align_to_codebook(_expand_phrase_tokens(words))   # defined above
        counts[a["status"]] = counts.get(a["status"], 0) + 1
        rows.append({
            "topic_id":         i,
            "alignment_status": a["status"],
            "domain_code":      a["domain_code"],
            "subtheme_code":    a["subtheme_code"],
            "overlap":          a["overlap"],
            "matched_terms":    a["matched_terms"],
            "top_words":        " | ".join(words),
        })
    return counts, rows


# ════════════════════════════════════════════════════════════════════════════════
# MODEL FITTERS (comparison-local, so RAKE injection + NMF coherence are controlled)
# ════════════════════════════════════════════════════════════════════════════════
def _fit_nmf(joined_texts: list[str], k: int,
             token_pattern: str, ngram_range: tuple[int, int]
             ) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """TF-IDF + NMF. token_pattern/ngram_range differ between the baseline and the
    RAKE-preprocessing config (the latter allows underscore phrase tokens)."""
    vec = TfidfVectorizer(
        max_features=MAX_VOCAB, min_df=5, max_df=0.85,
        ngram_range=ngram_range, token_pattern=token_pattern,
    )
    X = vec.fit_transform(joined_texts)
    nmf = NMF(n_components=k, random_state=RANDOM_SEED,
              max_iter=500, init="nndsvda")
    W = nmf.fit_transform(X)
    H = nmf.components_
    return W, H, vec.get_feature_names_out().tolist()


def _fit_lda(token_lists: list[list[str]], k: int
             ) -> tuple[LdaModel, corpora.Dictionary, list, list[list[str]]]:
    """
    BoW + LDA. Returns the model, the dictionary, the corpus, and the non-empty
    token lists actually used (so NMF coherence can be scored on the SAME
    dictionary + texts for a fair within-config comparison).
    """
    texts = [t for t in token_lists if len(t) > 0]
    dictionary = corpora.Dictionary(texts)
    dictionary.filter_extremes(no_below=5, no_above=0.85, keep_n=MAX_VOCAB)
    corpus = [dictionary.doc2bow(t) for t in texts]
    model = LdaModel(
        corpus=corpus, id2word=dictionary, num_topics=k,
        random_state=RANDOM_SEED, passes=LDA_PASSES, iterations=LDA_ITER,
        alpha="auto", eta="auto", per_word_topics=False,
    )
    return model, dictionary, corpus, texts


# ════════════════════════════════════════════════════════════════════════════════
# RAKE AS PREPROCESSING — phrase injection
# ════════════════════════════════════════════════════════════════════════════════
def _corpus_rake_phrases(raw_texts: list[str], top_n: int = RAKE_VOCAB_N) -> list[str]:
    """Top corpus-level RAKE phrases, restricted to genuine multi-word phrases."""
    scored = rake_keywords(raw_texts, top_n=top_n * 3)   # defined above
    phrases = [p for p, _ in scored if len(p.split()) >= 2]
    return phrases[:top_n]


def _inject_rake_phrases(raw_texts: list[str],
                         token_lists: list[list[str]],
                         joined_texts: list[str],
                         phrases: list[str]
                         ) -> tuple[list[list[str]], list[str]]:
    """
    For every document, append an underscore-joined token for each RAKE phrase
    found in its cleaned text (whole-phrase match). The phrase becomes one atomic
    feature for both LDA (token list) and NMF (joined string), sitting alongside
    the original unigrams rather than replacing them.
    """
    import re as _re
    # Pre-compile a word-boundary matcher per phrase; longer phrases first so the
    # most specific multi-word match wins where phrases nest.
    phrases_sorted = sorted(phrases, key=lambda p: -len(p))
    compiled = [(p, "_".join(p.split()),
                 _re.compile(r"\b" + _re.escape(p) + r"\b"))
                for p in phrases_sorted]

    aug_tokens: list[list[str]] = []
    aug_joined: list[str] = []
    for raw, toks, joined in zip(raw_texts, token_lists, joined_texts):
        cleaned = clean_text(str(raw))            # defined above
        extra = [merged for (_p, merged, rgx) in compiled if rgx.search(cleaned)]
        aug_tokens.append(list(toks) + extra)
        aug_joined.append((joined + " " + " ".join(extra)).strip())
    return aug_tokens, aug_joined


# ════════════════════════════════════════════════════════════════════════════════
# RAKE AS POST-PROCESSING — topic labeling
# ════════════════════════════════════════════════════════════════════════════════
def _nmf_doc_topics(W: np.ndarray) -> np.ndarray:
    return W.argmax(axis=1)


def _lda_doc_topics(model: LdaModel, dictionary: corpora.Dictionary,
                    token_lists: list[list[str]]) -> list[int]:
    """Dominant LDA topic per document, aligned to the token_lists order."""
    dom = []
    for toks in token_lists:
        bow = dictionary.doc2bow(toks)
        if not bow:
            dom.append(-1)
            continue
        dist = model.get_document_topics(bow, minimum_probability=0.0)
        dom.append(max(dist, key=lambda x: x[1])[0])
    return dom


def _rake_label_topics(raw_texts: list[str], doc_topics, k: int) -> dict[int, str]:
    """
    Group documents by their dominant topic and run RAKE on each group; the
    top-scoring phrase(s) become that topic's label. This is the config-3 product:
    RAKE naming topics that were discovered without it.
    """
    by_topic: dict[int, list[str]] = {t: [] for t in range(k)}
    for raw, t in zip(raw_texts, doc_topics):
        if 0 <= int(t) < k:
            by_topic[int(t)].append(str(raw))
    labels: dict[int, str] = {}
    for t in range(k):
        docs = by_topic[t]
        if len(docs) < 5:
            labels[t] = ""
            continue
        phrases = rake_keywords(docs, top_n=RAKE_LABEL_N)   # defined above
        labels[t] = " / ".join(p for p, _ in phrases)
    return labels


# ════════════════════════════════════════════════════════════════════════════════
# FIGURES
# ════════════════════════════════════════════════════════════════════════════════
_CFG_ORDER = ["rake_preprocess", "no_rake", "rake_postprocess"]
_CFG_NICE  = {
    "rake_preprocess":  "RAKE\npre-processing",
    "no_rake":          "No RAKE\n(baseline)",
    "rake_postprocess": "RAKE\npost-processing",
}
_MODEL_C = {"LDA": "#0d6efd", "NMF": "#198754"}


def _fig_coherence(metrics: pd.DataFrame, fig_dir: str):
    configs = [c for c in _CFG_ORDER if c in set(metrics["config"])]
    x = np.arange(len(configs))
    w = 0.38
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.set_facecolor(SF)
    for off, model in zip((-w / 2, w / 2), ("LDA", "NMF")):
        vals = [float(metrics[(metrics["config"] == c) &
                              (metrics["model"] == model)]["coherence_cv"].iloc[0])
                for c in configs]
        bars = ax.bar(x + off, vals, w, label=model, color=_MODEL_C[model],
                      edgecolor="white", zorder=3)
        for b, v in zip(bars, vals):
            if v == v:   # not NaN
                ax.annotate(f"{v:.3f}", (b.get_x() + b.get_width() / 2, v),
                            textcoords="offset points", xytext=(0, 4),
                            ha="center", fontsize=8, color=TX)
    ax.set_xticks(x)
    ax.set_xticklabels([_CFG_NICE[c] for c in configs], fontsize=9)
    ax.set_ylabel("Topic coherence (c_v)", fontsize=9, color=MU)
    ax.set_title(f"LDA vs NMF coherence at k={CMP_K} (higher = better)",
                 fontsize=11, color=TX, pad=12)
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "cmp_coherence.png"), **SAVE)
    plt.close()


def _fig_alignment(metrics: pd.DataFrame, fig_dir: str):
    configs = [c for c in _CFG_ORDER if c in set(metrics["config"])]
    rows = [(c, m) for c in configs for m in ("LDA", "NMF")]
    labels = [f"{_CFG_NICE[c].replace(chr(10), ' ')}\n{m}" for c, m in rows]
    aligned = [int(metrics[(metrics["config"] == c) & (metrics["model"] == m)]
                   ["n_aligned"].iloc[0]) for c, m in rows]
    ambig = [int(metrics[(metrics["config"] == c) & (metrics["model"] == m)]
                 ["n_ambiguous"].iloc[0]) for c, m in rows]
    induct = [int(metrics[(metrics["config"] == c) & (metrics["model"] == m)]
                  ["n_inductive"].iloc[0]) for c, m in rows]
    x = np.arange(len(rows))
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.set_facecolor(SF)
    ax.bar(x, aligned, label="aligned", color="#198754", edgecolor="white", zorder=3)
    ax.bar(x, ambig, bottom=aligned, label="ambiguous", color="#fd7e14",
           edgecolor="white", zorder=3)
    ax.bar(x, induct, bottom=np.array(aligned) + np.array(ambig),
           label="inductive", color="#adb5bd", edgecolor="white", zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Number of topics", fontsize=9, color=MU)
    ax.set_title(f"Codebook alignment of discovered topics (k={CMP_K})",
                 fontsize=11, color=TX, pad=12)
    ax.legend(frameon=False, fontsize=9, ncol=3)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "cmp_alignment.png"), **SAVE)
    plt.close()


def _fig_agreement(agreement: pd.DataFrame, fig_dir: str):
    summary = (agreement.groupby("config")["jaccard"].mean()
               .reindex([c for c in _CFG_ORDER if c in set(agreement["config"])]))
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.set_facecolor(SF)
    bars = ax.bar([_CFG_NICE[c] for c in summary.index], summary.values,
                  color="#6f42c1", edgecolor="white", zorder=3, width=0.55)
    for b, v in zip(bars, summary.values):
        ax.annotate(f"{v:.3f}", (b.get_x() + b.get_width() / 2, v),
                    textcoords="offset points", xytext=(0, 4),
                    ha="center", fontsize=9, color=TX)
    ax.set_ylabel("Mean LDA↔NMF top-word Jaccard", fontsize=9, color=MU)
    ax.set_ylim(0, max(0.4, float(summary.max()) * 1.25) if len(summary) else 0.4)
    ax.set_title(f"Cross-method agreement at k={CMP_K} (higher = more convergent)",
                 fontsize=11, color=TX, pad=12)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "cmp_agreement.png"), **SAVE)
    plt.close()


# ════════════════════════════════════════════════════════════════════════════════
# ORCHESTRATOR
# ════════════════════════════════════════════════════════════════════════════════
def compare_lda_nmf(df: pd.DataFrame,
                    token_lists: list[list[str]],
                    joined_texts: list[str],
                    k: int = CMP_K,
                    output_root: str | None = None,
                    top_n: int = TOP_N_WORDS) -> dict:
    """
    Compare LDA vs NMF at a single k across the three RAKE configurations and
    write all comparison artefacts. Returns the metrics DataFrame (also saved).
    """
    if output_root is None:
        output_root = OUTPUT_ROOT          # defined above
    out_dir = os.path.join(output_root, f"k{k}_lda_nmf_comparison")
    fig_dir = os.path.join(out_dir, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    raw_texts = df["comment_body"].astype(str).tolist()

    print(f"\n{'#'*70}")
    print(f"# LDA vs NMF comparison @ k={k}   ->  {out_dir}")
    print(f"{'#'*70}")

    metric_rows: list[dict] = []
    keyword_rows: list[dict] = []
    agreement_rows: list[dict] = []
    align_detail_rows: list[dict] = []

    # ── Build the three feature configurations ────────────────────────────────
    # Baseline tokens/joined are reused for config 2 AND config 3 (post-processing
    # labels baseline models), so only config 1 needs the RAKE-augmented inputs.
    print("\nExtracting corpus-level RAKE phrases for the pre-processing config...")
    rake_vocab = _corpus_rake_phrases(raw_texts, top_n=RAKE_VOCAB_N)
    print(f"  {len(rake_vocab)} multi-word RAKE phrases injected as atomic tokens")
    aug_tokens, aug_joined = _inject_rake_phrases(
        raw_texts, token_lists, joined_texts, rake_vocab)

    configs = {
        # config: (token_lists, joined_texts, nmf_token_pattern, nmf_ngram_range)
        "rake_preprocess": (aug_tokens, aug_joined,
                            r"(?u)\b[a-z][a-z_]*[a-z]\b", (1, 1)),
        "no_rake":         (token_lists, joined_texts,
                            r"[a-z][a-z]+", (1, 2)),
    }

    # Cache the baseline models so config 3 can reuse them for labeling.
    baseline_cache: dict = {}

    for cfg_name, (toks, joined, tok_pat, ngram) in configs.items():
        print(f"\n── config: {cfg_name} ──")

        # LDA
        lda_model, lda_dict, lda_corpus, lda_texts = _fit_lda(toks, k)
        lda_words = _lda_top_words(lda_model, n=top_n)
        lda_coh = _coherence_cv(lda_words, lda_texts, lda_dict)

        # NMF (scored on the SAME dictionary+texts as LDA for a fair comparison)
        W, H, feat = _fit_nmf(joined, k, token_pattern=tok_pat, ngram_range=ngram)
        nmf_words = get_top_words(H, feat, n=top_n)         # defined above
        nmf_coh = _coherence_cv(nmf_words, lda_texts, lda_dict)

        print(f"  LDA c_v={lda_coh:.4f}   NMF c_v={nmf_coh:.4f}")

        # Codebook alignment per model
        lda_counts, lda_align = _alignment_counts(lda_words)
        nmf_counts, nmf_align = _alignment_counts(nmf_words)

        # Cross-method agreement (LDA vs NMF)
        mean_j, pairs = _match_topics(lda_words, nmf_words)
        print(f"  mean LDA<->NMF Jaccard = {mean_j:.3f}")

        for model, words, coh, counts in (
            ("LDA", lda_words, lda_coh, lda_counts),
            ("NMF", nmf_words, nmf_coh, nmf_counts),
        ):
            metric_rows.append({
                "config":          cfg_name,
                "model":           model,
                "k":               k,
                "coherence_cv":    round(coh, 4) if coh == coh else float("nan"),
                "topic_diversity": _topic_diversity(words, n=top_n),
                "n_aligned":       counts.get("aligned", 0),
                "n_ambiguous":     counts.get("ambiguous", 0),
                "n_inductive":     counts.get("inductive_candidate", 0),
                "mean_cross_method_jaccard": mean_j,
            })
            for i, w in enumerate(words):
                keyword_rows.append({
                    "config": cfg_name, "model": model,
                    "topic_id": i, "top_words": " | ".join(w),
                })

        for r in lda_align:
            align_detail_rows.append({"config": cfg_name, "model": "LDA", **r})
        for r in nmf_align:
            align_detail_rows.append({"config": cfg_name, "model": "NMF", **r})
        for p in pairs:
            agreement_rows.append({"config": cfg_name, **p})

        baseline_cache[cfg_name] = {
            "lda_model": lda_model, "lda_dict": lda_dict,
            "lda_texts_full": toks, "W": W,
        }

    # ── Config 3: RAKE as post-processing (topic labeling) ────────────────────
    # Re-uses the no_rake baseline models; RAKE only NAMES the topics here.
    print("\n── config: rake_postprocess (topic labeling on no_rake baseline) ──")
    base = baseline_cache["no_rake"]
    nmf_dom = _nmf_doc_topics(base["W"])
    lda_dom = _lda_doc_topics(base["lda_model"], base["lda_dict"], token_lists)

    nmf_labels = _rake_label_topics(raw_texts, nmf_dom, k)
    lda_labels = _rake_label_topics(raw_texts, lda_dom, k)

    # Pull the no_rake top words back for context next to each RAKE label.
    nmf_kw = {r["topic_id"]: r["top_words"]
              for r in keyword_rows
              if r["config"] == "no_rake" and r["model"] == "NMF"}
    lda_kw = {r["topic_id"]: r["top_words"]
              for r in keyword_rows
              if r["config"] == "no_rake" and r["model"] == "LDA"}

    label_rows = []
    for t in range(k):
        label_rows.append({
            "topic_id":       t,
            "nmf_rake_label": nmf_labels.get(t, ""),
            "nmf_top_words":  nmf_kw.get(t, ""),
            "lda_rake_label": lda_labels.get(t, ""),
            "lda_top_words":  lda_kw.get(t, ""),
        })
    labels_df = pd.DataFrame(label_rows)

    # Coverage = share of topics RAKE could label (had enough docs / phrases).
    nmf_cov = round(np.mean([1 if nmf_labels.get(t) else 0 for t in range(k)]), 3)
    lda_cov = round(np.mean([1 if lda_labels.get(t) else 0 for t in range(k)]), 3)
    print(f"  RAKE label coverage — NMF {nmf_cov:.0%} · LDA {lda_cov:.0%}")

    # Record config-3 as metric rows too (coherence/alignment carried over from
    # the no_rake baseline, since the MODELS are identical — only labeling differs;
    # the distinguishing metric is RAKE label coverage).
    for model, cov in (("LDA", lda_cov), ("NMF", nmf_cov)):
        base_metric = next(m for m in metric_rows
                           if m["config"] == "no_rake" and m["model"] == model)
        metric_rows.append({
            "config":          "rake_postprocess",
            "model":           model,
            "k":               k,
            "coherence_cv":    base_metric["coherence_cv"],
            "topic_diversity": base_metric["topic_diversity"],
            "n_aligned":       base_metric["n_aligned"],
            "n_ambiguous":     base_metric["n_ambiguous"],
            "n_inductive":     base_metric["n_inductive"],
            "mean_cross_method_jaccard": base_metric["mean_cross_method_jaccard"],
            "rake_label_coverage": cov,
        })

    # ── Assemble + write ──────────────────────────────────────────────────────
    metrics = pd.DataFrame(metric_rows)
    # Stable config ordering for readability
    metrics["config"] = pd.Categorical(metrics["config"],
                                        categories=_CFG_ORDER, ordered=True)
    metrics = metrics.sort_values(["config", "model"]).reset_index(drop=True)

    keywords = pd.DataFrame(keyword_rows)
    agreement = pd.DataFrame(agreement_rows)
    align_detail = pd.DataFrame(align_detail_rows)

    metrics.to_csv(os.path.join(out_dir, "comparison_metrics.csv"), index=False)
    keywords.to_csv(os.path.join(out_dir, "topic_keywords_by_model.csv"), index=False)
    agreement.to_csv(os.path.join(out_dir, "cross_method_agreement.csv"), index=False)
    align_detail.to_csv(os.path.join(out_dir, "topic_codebook_alignment.csv"),
                        index=False)
    labels_df.to_csv(os.path.join(out_dir, "topic_labels_rake.csv"),
                     index=False, encoding="utf-8-sig")

    # ── Figures ────────────────────────────────────────────────────────────────
    _fig_coherence(metrics, fig_dir)
    _fig_alignment(metrics, fig_dir)
    _fig_agreement(agreement, fig_dir)

    # ── Console summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"COMPARISON COMPLETE (k={k})  ->  {out_dir}")
    print(f"{'='*70}")
    show = metrics[["config", "model", "coherence_cv", "topic_diversity",
                    "n_aligned", "n_ambiguous", "n_inductive",
                    "mean_cross_method_jaccard"]]
    with pd.option_context("display.width", 120, "display.max_columns", None):
        print(show.to_string(index=False))
    print("\nFiles written:")
    for f in ("comparison_metrics.csv", "topic_keywords_by_model.csv",
              "cross_method_agreement.csv", "topic_codebook_alignment.csv",
              "topic_labels_rake.csv",
              "figures/cmp_coherence.png", "figures/cmp_alignment.png",
              "figures/cmp_agreement.png"):
        print(f"  {os.path.join(out_dir, f)}")

    return {"out_dir": out_dir, "metrics": metrics}

# ════════════════════════════════════════════════════════════════════════════════
# RUN
# ════════════════════════════════════════════════════════════════════════════════
def _ensure_data_dir(data_dir: str) -> None:
    """On Colab, mount Google Drive if the dataset folder isn't visible yet."""
    if os.path.exists(data_dir):
        return
    try:
        from google.colab import drive   # only present on Colab
        if not os.path.ismount("/content/drive"):
            print("Dataset folder not found yet — mounting Google Drive...")
            drive.mount("/content/drive")
    except Exception:
        pass   # not on Colab, or user declined the mount


if __name__ == "__main__":
    # If you ran the main pipeline cell, its df/token_lists/joined_texts are LOCAL
    # to main() and are NOT visible here, so this cell normally loads the data
    # itself. To reuse already-loaded data instead, expose it as globals first:
    #     df = load_data(); token_lists, joined_texts = preprocess(df)
    # and then run this cell.
    try:
        _df, _tok, _joined = df, token_lists, joined_texts          # noqa: F821
    except NameError:
        _ensure_data_dir(DATA_DIR)
        if not os.path.exists(DATA_DIR):
            raise FileNotFoundError(
                f"DATA_DIR not found:\n  {DATA_DIR}\n\n"
                "Fix one of these, then re-run:\n"
                "  • Mount Google Drive (Colab):\n"
                "        from google.colab import drive; drive.mount('/content/drive')\n"
                "  • Or edit DATA_DIR at the top of this cell to the folder that\n"
                "    holds cleaned/comments_clean.csv (or dataset_comments.csv)."
            )
        _df = load_data()
        _tok, _joined = preprocess(_df)
    compare_lda_nmf(_df, _tok, _joined, k=15, output_root=OUTPUT_ROOT)

